# Pretraining and Fine-tuning in Transformers

This notebook covers how transformer models learn language — from pretraining on massive corpora to being fine-tuned for specific tasks like sentiment analysis or Q&A.

---


## 🧠 What is Pretraining?

In pretraining, the model is trained on **unlabeled text** using self-supervised objectives.

### Common Pretraining Objectives:
- **Masked Language Modeling (MLM)** — BERT
  > Predict missing tokens from context  
  > Example: `"The cat [MASK] on the mat."`

- **Causal Language Modeling (CLM)** — GPT
  > Predict the next word  
  > Example: `"The cat sat on"`

- **Seq2Seq Learning** — T5/BART
  > Input and output are both text (e.g., translation)

---


### 🧪 Example: Masked Language Modeling with BERT

In [1]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

input_text = "The capital of France is [MASK]."
inputs = tokenizer(input_text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
masked_index = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1].item()
predicted_token_id = logits[0, masked_index].argmax(dim=-1).item()

print("Predicted word:", tokenizer.decode([predicted_token_id]))


/Users/rahulsaini/Documents/Repositories/gen-ai-cookbook/04-gen-ai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Predicted word: paris


## 🛠️ What is Fine-tuning?

After pretraining, the model can be **fine-tuned** on a smaller dataset for a specific task:

- **Classification** (e.g., sentiment, spam detection)
- **Question Answering**
- **Summarization**
- **NER** (Named Entity Recognition)

Fine-tuning requires labeled data and adapts the pretrained weights to the target task.

---


## 🏗️ Fine-tuning Architecture

We add a **task-specific head** (like a classifier) on top of the pretrained model:

- For classification: a linear layer over the [CLS] token
- For QA: two linear layers predicting start and end positions

```python
from transformers import BertForSequenceClassification
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
```

---


In [2]:
from transformers import pipeline

clf = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
print(clf("I absolutely loved the movie!"))


Device set to use mps:0


[{'label': 'POSITIVE', 'score': 0.9998770952224731}]


## ✅ Summary

- Pretraining teaches the model to understand language
- Fine-tuning teaches it to perform specific tasks
- Hugging Face makes both steps accessible with pretrained models and pipelines

---
